In [1]:
import sys, os
import re
import gc
import glob
import pandas as pd
import ast
import itertools
import numpy as np
from pathlib import Path

In [2]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [3]:
from tqdm.notebook import tqdm

tqdm.pandas()

In [4]:
import warnings
warnings.filterwarnings("ignore")

### Loading package

In [5]:
import sys
from pathlib import Path

here_path = Path().resolve()
repo_path = here_path.parents[1]
sys.path.append(str(repo_path))

In [6]:
from py.utils import verifyDir,verifyFile

In [7]:
from py.config import Config

cfg = Config()

np.random.seed(cfg.RANDOM_STATE)
cfg.DATA_PATH, cfg.MODEL_PATH

('/media/felipe/DATA21/datasets/', '/media/felipe/DATA21/models/')

### Loading data

In [8]:
DATA_PATH=f"{cfg.DATA_PATH}crimebb/"
CSV_PATH = f"{DATA_PATH}/{cfg.YEAR}/csv/"
HF_PROCESSED = f"{DATA_PATH}/{cfg.YEAR}/csv/topics/{cfg.TOPIC_SEARCH}/"
FEATURES_PATH = f"{cfg.MODEL_PATH}crimebb/{cfg.YEAR}/features/"

In [9]:
verifyDir(FEATURES_PATH)

### Settup variables

In [10]:
n_grams = (1,1) # (number of words, number of strides (jumps) )
num_words = 5000
min_df=0.01
max_df=0.99

dim_vec = 5000
min_count=10
window=15
sample=6e-5 
alpha=0.001
min_alpha=0.0007
negative=1

## Loading data

In [11]:
data_df = pd.read_csv(f"{HF_PROCESSED}/HackForums_Languages.csv", sep="\t", low_memory=False, encoding='utf-8')
data_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067 entries, 0 to 1066
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   thread_id             1067 non-null   int64  
 1   annotations           1067 non-null   str    
 2   site_id               1067 non-null   int64  
 3   site_name             1067 non-null   str    
 4   board_id              1067 non-null   int64  
 5   board_title           1067 non-null   str    
 6   thread_title          1067 non-null   str    
 7   content               1067 non-null   str    
 8   cve_codes_list        1067 non-null   str    
 9   cve_unique_list       1067 non-null   str    
 10  r_IMG                 1067 non-null   str    
 11  r_CITING              1067 non-null   str    
 12  r_IFRAME              1067 non-null   str    
 13  r_LINK                1067 non-null   str    
 14  r_CODE                1067 non-null   str    
 15  r_ATTACHMENT          1067 non-n

#### Processing texts

In [12]:
data_df["lang_detected"].value_counts()

lang_detected
english       1050
not found       13
portuguese       3
russian          1
Name: count, dtype: int64

In [13]:
# LANGUAGE_TO_EVAL=[lang for lang in data_df["lang_detected"].unique() if lang!="not found"]
LANGUAGE_TO_EVAL=["english"]

In [14]:
filter_df = data_df[data_df["lang_detected"].isin(LANGUAGE_TO_EVAL)].copy()
filter_df.reset_index(inplace=True, drop=True)
filter_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1050 entries, 0 to 1049
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   thread_id             1050 non-null   int64  
 1   annotations           1050 non-null   str    
 2   site_id               1050 non-null   int64  
 3   site_name             1050 non-null   str    
 4   board_id              1050 non-null   int64  
 5   board_title           1050 non-null   str    
 6   thread_title          1050 non-null   str    
 7   content               1050 non-null   str    
 8   cve_codes_list        1050 non-null   str    
 9   cve_unique_list       1050 non-null   str    
 10  r_IMG                 1050 non-null   str    
 11  r_CITING              1050 non-null   str    
 12  r_IFRAME              1050 non-null   str    
 13  r_LINK                1050 non-null   str    
 14  r_CODE                1050 non-null   str    
 15  r_ATTACHMENT          1050 non-n

In [15]:
from py.nlpToolkit import TextProcesser

txt_processer = TextProcesser(language_to_process="english", 
                              keyword_sep='famveer',
                              keep_emojis=True,
                              #exclude_pipe=['morphologizer', "parser", "ner", "attribute_ruler", "tagger"])
                              exclude_pipe=['tagger', 'parser', 'ner'])
print(txt_processer.get_nlp_pipeline())

([('tok2vec', <spacy.pipeline.tok2vec.Tok2Vec object at 0x75ce46917e90>), ('text_filter', <py.nlpToolkit.nlpToolkit.text_processing.text_filter.TextFilter object at 0x75ce45d99370>), ('attribute_ruler', <spacy.pipeline.attributeruler.AttributeRuler object at 0x75ce4672e050>), ('lemmatizer', <spacy.lang.en.lemmatizer.EnglishLemmatizer object at 0x75ce4672ee50>)], ['tok2vec', 'text_filter', 'attribute_ruler', 'lemmatizer'])


In [16]:
%%time
text_processed = txt_processer.process_texts_by_language(filter_df["content"].values)

Preprocessing texts:   0%|          | 0/1050 [00:00<?, ?it/s]

CPU times: user 2.96 s, sys: 255 ms, total: 3.21 s
Wall time: 22 s


In [17]:
filter_df["content_processed"] = text_processed

In [18]:
filter_df["content_length"] = filter_df["content"].apply(lambda x: len(x.split()) )

### Extracting Features

Text cleaning and processing:

* occurred in too many documents (max_df)
* occurred in too few documents (min_df)
* were cut off by feature selection (max_features).
* number of words to evaluate context (ngram)

In [19]:
# from sklearn.preprocessing import LabelEncoder
# le = LabelEncoder()
# le.fit(topics_of_interest)
# le.transform(topics_of_interest)

### Dictionary

In [20]:
filter_df["annotations_class"] = filter_df["annotations"].astype('category').cat.codes

In [21]:
dict_features = {}
dict_features["content"] = filter_df["content"].values
dict_features["content_processed"] = filter_df["content_processed"].values
dict_features["annotations"] = filter_df["annotations"].values
dict_features["annotations_class"] = filter_df["annotations_class"].values

In [22]:
if cfg.PROCESS_TXT:
    documents = filter_df["content_processed"].values
    out_filename="_processed"
else:
    documents = filter_df["content"].values
    out_filename=""

In [23]:
text_input = [[word for word in doc.split()] for doc in documents]

In [24]:
# Create Dictionary
import gensim.corpora as corpora
id2word = corpora.Dictionary(text_input)

In [25]:
# Create Corpus: Term Document Frequency
corpus = [id2word.doc2bow(text) for text in text_input]

In [26]:
dict_features["corpus"] = corpus
dict_features["id2word"] = id2word

### TF-IDF

Term frequency (TF) = $\large \frac{\text{(Number of Occurrences of a word)}}{\text{(Total words in the document)}}$

Inverse Document Frequency (IDF) word = $\large \log(\frac{\text{(Total words in the document)}}{\text{(Number of documents containing the word))}}$

$TF-IDF_{score} = TF \times IDF$

In [27]:
from sklearn.feature_extraction.text import TfidfTransformer, TfidfVectorizer

In [28]:
tf_idf = TfidfVectorizer(max_features=num_words, # features: most occurring words
                         ngram_range=n_grams, # len of words context
                         sublinear_tf=True, # sublinear tf scaler
                         min_df=min_df, # min_df: a word repeated at least in 5 docs
                         max_df=max_df, # max_df: only those words that occur in a maximum of 90% of all the documents
                        )

In [29]:
%%time
X_tfidf = tf_idf.fit_transform(documents).toarray()
X_tfidf.shape, X_tfidf.dtype

CPU times: user 76 ms, sys: 2.84 ms, total: 78.8 ms
Wall time: 79.2 ms


((1050, 1773), dtype('float64'))

In [30]:
X_tfidf[0][X_tfidf[0]>0]

array([0.45652709, 0.14578941, 0.24731878, 0.38199147, 0.28435155,
       0.53163925, 0.31293569, 0.31909178])

In [31]:
dict_features["X_tfidf"] = X_tfidf
dict_features["tfidf_names"] = tf_idf.get_feature_names_out()

### Word2Vec - Bag of Words

In [32]:
from sklearn.feature_extraction.text import CountVectorizer

In [33]:
cbow = CountVectorizer(max_features=num_words, # features: most occurring words
                       ngram_range=n_grams, # len of words context
                       min_df=min_df, # min_df: a word repeated at least in 5 docs
                       max_df=max_df, # max_df: only those words that occur in a maximum of 90% of all the documents
                      )

In [34]:
%%time
X_cbow = cbow.fit_transform(documents).toarray()
X_cbow.shape, X_cbow.dtype

CPU times: user 77.7 ms, sys: 2.99 ms, total: 80.7 ms
Wall time: 80 ms


((1050, 1773), dtype('int64'))

In [35]:
X_cbow[0][X_cbow[0]>0]

array([1, 1, 1, 1, 1, 1, 1, 1])

In [36]:
dict_features["X_cbow"] = X_cbow
dict_features["cbow_names"] = cbow.get_feature_names_out()

### Word2Vec - Skip Gram

In [37]:
from gensim.models.phrases import Phrases

In [38]:
word_list_per_doc = [row.split() for row in documents]

In [39]:
phrases = Phrases(word_list_per_doc, min_count=30, progress_per=10000)

In [40]:
from gensim.models.phrases import Phraser

In [41]:
bigram = Phraser(phrases)

In [42]:
sentences = bigram[word_list_per_doc]

In [43]:
from gensim.models import Word2Vec

In [44]:
skip_gram = Word2Vec(min_count=20, # word frequency
                      vector_size=dim_vec, #output vector
                      window=15, # distance between words
                      sg=1, #Skip-Gram
                      sample=6e-5, 
                      alpha=0.03, 
                      min_alpha=0.0007, 
                      negative=20)

In [45]:
skip_gram.build_vocab(sentences, progress_per=10000)

In [46]:
skip_gram.train(sentences, total_examples=skip_gram.corpus_count, epochs=skip_gram.epochs)

(384486, 1327390)

In [47]:
skip_gram.wv.most_similar('vulnerability', topn=20)

[('exploited', 0.9493862390518188),
 ('execute_arbitrary', 0.9451543688774109),
 ('code_execution', 0.9366374015808105),
 ('remote_code', 0.9347350597381592),
 ('flaw', 0.9335150122642517),
 ('advisory', 0.9311370849609375),
 ('affects', 0.9296135902404785),
 ('attackers', 0.9279031753540039),
 ('discovered', 0.9247593879699707),
 ('buffer_overflow', 0.924311637878418),
 ('lead', 0.9191014766693115),
 ('remotely', 0.910603940486908),
 ('affected', 0.910099983215332),
 ('execution', 0.9089442491531372),
 ('flaws', 0.9083291292190552),
 ('restrictions', 0.9081501960754395),
 ('arbitrary_code', 0.9039666652679443),
 ('crafted', 0.9017512202262878),
 ('specially', 0.8991050720214844),
 ('denial_service', 0.8990381956100464)]

In [48]:
skip_gram.wv.most_similar('exploit', topn=20)

[('silent', 0.9341872334480286),
 ('silent_javascript', 0.9228498339653015),
 ('tips_viewer', 0.9226953983306885),
 ('generator_trillium', 0.9193177223205566),
 ('unique_powershell', 0.9162746071815491),
 ('password_generator', 0.91554856300354),
 ('en_de', 0.915128767490387),
 ('silent_encoded', 0.9149667024612427),
 ('unique_shellcode', 0.9144242405891418),
 ('javascript', 0.9143040776252747),
 ('word_autoclose', 0.9124715924263),
 ('screen_capture', 0.911440908908844),
 ('silent_shortcut', 0.9113404750823975),
 ('javascript_picture', 0.9110262989997864),
 ('silent_chm', 0.9091628789901733),
 ('account_manager', 0.9085859656333923),
 ('excel_properties', 0.9080346822738647),
 ('cve', 0.9059658050537109),
 ('excel_ole', 0.9059264659881592),
 ('v6_edge', 0.905532717704773)]

In [49]:
skip_gram.wv.most_similar('fud', topn=20)

[('kapo', 0.9517136812210083),
 ('undetected', 0.9509088397026062),
 ('vouch', 0.9445192813873291),
 ('macro', 0.941116988658905),
 ('bought', 0.9349544048309326),
 ('protected_view', 0.932598352432251),
 ('buy', 0.928255021572113),
 ('sir', 0.9280887842178345),
 ('paid', 0.9279965758323669),
 ('dude', 0.9265270829200745),
 ('vouches', 0.9219868183135986),
 ('confusion', 0.9201355576515198),
 ('jdb', 0.9193494319915771),
 ('add_skype', 0.9183883666992188),
 ('sale', 0.9154298305511475),
 ('omeb', 0.915178656578064),
 ('ole', 0.9141475558280945),
 ('guys', 0.9139816761016846),
 ('reply', 0.9133324027061462),
 ('working', 0.9131550788879395)]

In [61]:
skip_gram.wv.most_similar("cve", topn=20)

[('office', 0.9120556116104126),
 ('exploit', 0.9059657454490662),
 ('pdf', 0.8918981552124023),
 ('security', 0.8905009031295776),
 ('trillium', 0.8854718804359436),
 ('v6', 0.8712738156318665),
 ('multisploit_tool', 0.8708304166793823),
 ('exploit_generator', 0.868497908115387),
 ('regards', 0.8681147694587708),
 ('silent_doc', 0.8596176505088806),
 ('word_ooxml', 0.8590173721313477),
 ('hi', 0.855878472328186),
 ('coming_soon', 0.8496749401092529),
 ('avaible_teamviewer', 0.8465157747268677),
 ('excel', 0.846287727355957),
 ('tips_viewer', 0.8448569774627686),
 ('silent', 0.8444298505783081),
 ('en_de', 0.8418197631835938),
 ('support_requests', 0.8416229486465454),
 ('videos', 0.8402026295661926)]

In [62]:
skip_gram.wv.most_similar("poc", topn=20)

[('nessus', 0.9168578386306763),
 ('paste', 0.9141659140586853),
 ('exploitable', 0.9049966335296631),
 ('helps', 0.9018920063972473),
 ('exactly', 0.9009802937507629),
 ('knows', 0.9003925323486328),
 ('easier', 0.8980070352554321),
 ('basically', 0.8972158432006836),
 ('pretty', 0.8959841132164001),
 ('stupid', 0.89361572265625),
 ('obviously', 0.8930208086967468),
 ('alert', 0.8883610963821411),
 ('github', 0.8879958987236023),
 ('trying', 0.8863307237625122),
 ('gives', 0.8859156966209412),
 ('guess', 0.8833742141723633),
 ('help', 0.8828095197677612),
 ('provided', 0.8823408484458923),
 ('escape', 0.8820659518241882),
 ('simply', 0.8755727410316467)]

### Doc2Vec

In [52]:
from gensim.test.utils import common_texts
from gensim.models.doc2vec import Doc2Vec, TaggedDocument

In [53]:
sentences_doc = [TaggedDocument(doc, [i]) for i, doc in enumerate(documents)]

In [54]:
doc2vec = Doc2Vec(vector_size=dim_vec, # output vector
                  min_count=min_count, # word frequency
                  window=window, # distance between words
                  sample=sample, # words with frequency lower than this value, are ignored
                  alpha=alpha, 
                  min_alpha=min_alpha, 
                  negative=negative # noisy words
                 )

In [55]:
doc2vec.build_vocab(sentences_doc, progress_per=10000)

In [56]:
doc2vec.train(sentences_doc, total_examples=doc2vec.corpus_count, epochs=doc2vec.epochs)

In [57]:
X_doc2vec = np.array([doc2vec.infer_vector([doc]) for doc in documents])
X_doc2vec.shape, X_doc2vec.dtype

((1050, 5000), dtype('float32'))

In [58]:
dict_features["X_doc2vec"] = X_doc2vec

### Saving Files

In [59]:
for k in dict_features.keys():
    print(k, len(dict_features[k]))

content 1050
content_processed 1050
annotations 1050
annotations_class 1050
corpus 1050
id2word 18237
X_tfidf 1050
tfidf_names 1773
X_cbow 1050
cbow_names 1773
X_doc2vec 1050


In [60]:
import pickle
# create a binary pickle file 
file_ = open(f"{FEATURES_PATH}metadata{out_filename}.pkl","wb")

# write the python object (dict) to pickle file
pickle.dump(dict_features, file_)

# close file
file_.close()